In [1]:
import torch
from master_model import MasterModel
from ttokenizer import Tokenizer

u_tokenizer = Tokenizer("tokenizer.json")
prompt = "the capital of united"
tokens = u_tokenizer.encode(prompt)

torch.manual_seed(1)
u_model = MasterModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings = u_model(tokens)
sentence_meanings.shape
#sentence_meanings_with_attention_context = u_model(tokens)
#sentence_meanings_with_attention_context

torch.Size([7, 4])

In [3]:
from transformers import Gemma3ForCausalLM

gemma_model = Gemma3ForCausalLM.from_pretrained("google/gemma-3-1b-it")


/Users/gonul/llm-from-scratch/.llm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 340/340 [00:00<00:00, 8822.47it/s]


In [4]:
u_model, gemma_model

(MasterModel(
   (embedding): Embedding(64, 4)
   (pos_embedding): Embedding(32, 4)
   (self_attention): SelfAttention(
     (q_weights): Linear(in_features=4, out_features=4, bias=False)
     (k_weights): Linear(in_features=4, out_features=4, bias=False)
     (v_weights): Linear(in_features=4, out_features=4, bias=False)
   )
 ),
 Gemma3ForCausalLM(
   (model): Gemma3TextModel(
     (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
     (layers): ModuleList(
       (0-25): 26 x Gemma3DecoderLayer(
         (self_attn): Gemma3Attention(
           (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
           (k_proj): Linear(in_features=1152, out_features=256, bias=False)
           (v_proj): Linear(in_features=1152, out_features=256, bias=False)
           (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
           (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
           (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
         )
       

In [9]:
attention_scores = sentence_meanings @ sentence_meanings.T #burda ilk sentence_meanings Q, sentence_meanings.T K oldu
attention_scores.shape

torch.Size([7, 7])

In [10]:
attention_weights = torch.softmax(attention_scores,dim=1)
attention_weights


tensor([[9.6005e-01, 7.7638e-03, 2.4494e-02, 9.7019e-04, 2.4756e-03, 4.0204e-03,
         2.2848e-04],
        [3.5447e-01, 2.2772e-01, 1.1067e-01, 9.7859e-02, 9.1879e-02, 8.4686e-02,
         3.2719e-02],
        [1.3174e-01, 1.3037e-02, 7.1425e-01, 2.9118e-02, 8.5876e-02, 1.6926e-02,
         9.0602e-03],
        [3.9171e-02, 8.6539e-02, 2.1859e-01, 2.0138e-01, 2.1989e-01, 8.6539e-02,
         1.4790e-01],
        [5.8228e-02, 4.7332e-02, 3.7555e-01, 1.2810e-01, 2.1774e-01, 6.9514e-02,
         1.0354e-01],
        [1.5512e-01, 7.1569e-02, 1.2143e-01, 8.2701e-02, 1.1403e-01, 1.9245e-01,
         2.6269e-01],
        [2.8782e-03, 9.0277e-03, 2.1221e-02, 4.6144e-02, 5.5456e-02, 8.5766e-02,
         7.7951e-01]], grad_fn=<SoftmaxBackward0>)

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V
$$

$$
Q = \text{Query (Sorgu)}
$$

$$
K = \text{Key (Anahtar)}
$$

$$
V = \text{Value (İçerik)}
$$

$$
QK^{\top} \rightarrow \text{Similarity (benzerlik)}
$$

$$
\sqrt{d_k} \rightarrow \text{Scaling (sayilar cok buyumesin)}
$$

$$
\text{softmax}(\cdot) \rightarrow \text{Attention weights}
$$

$$
\text{weights} \times V \rightarrow \text{Weighted sum}
$$

$$
d_k = \text{Key vektörünün boyutu}
$$

$$
\text{embedding\_dim} = 512
$$

$$
\text{heads} = 8
$$

$$
d_k = \frac{512}{8} = 64
$$

$$
\text{Neden } \sqrt{d_k} \text{ var?}
$$

$$
\text{Dot product büyür, softmax saturate olur (çok keskin olur).}
$$

$$
\text{Scale} = \frac{1}{\sqrt{d_k}}
$$

#q icin agirliklar(sentence i agirliklar ile carp)
#k icin agirliklar(k i agirliklar ile carp cumleden olusturcaz)
#v icin agirliklar((sentence i value agirliklar ile carp))
#q (from) -> k.v(to)
#herbir kelimenin birbirine ne kadar benzedigini anliyoruz.

In [ ]:
# new model
# SelfAttention added to MasterModel

# Casual Self Attention

In [ ]:
# bi token hemen kendinden sonraki kelime ile olan benzerligini bilmemeli, bunu maskelemek gerekir. amac zaten o kelimeyi tahmin etmek
# amac kendinden sonra gelen kelimeyi tahmin etmek. bunun icin maskeleme kullanicaz
#

In [5]:
# 7 tokenli cumle icin "the capital of united"
mask = torch.tril(torch.ones(attention_weights.shape[0], attention_weights.shape[0]))
mask

tensor([[1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1.]])

In [ ]:
#scaler carpim
attention_weights*mask

tensor([[0.9600, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3545, 0.2277, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1317, 0.0130, 0.7142, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0392, 0.0865, 0.2186, 0.2014, 0.0000, 0.0000, 0.0000],
        [0.0582, 0.0473, 0.3756, 0.1281, 0.2177, 0.0000, 0.0000],
        [0.1551, 0.0716, 0.1214, 0.0827, 0.1140, 0.1924, 0.0000],
        [0.0029, 0.0090, 0.0212, 0.0461, 0.0555, 0.0858, 0.7795]],
       grad_fn=<MulBackward0>)

In [14]:
torch.sum(attention_weights, dim=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)

In [17]:
mask = torch.tril(torch.ones(attention_weights.shape[0], attention_weights.shape[0]))

masked_attention_weights =  attention_weights.masked_fill(mask==0, -torch.inf)
masked_attention_weights

tensor([[0.9600,   -inf,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.3545, 0.2277,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1317, 0.0130, 0.7142,   -inf,   -inf,   -inf,   -inf],
        [0.0392, 0.0865, 0.2186, 0.2014,   -inf,   -inf,   -inf],
        [0.0582, 0.0473, 0.3756, 0.1281, 0.2177,   -inf,   -inf],
        [0.1551, 0.0716, 0.1214, 0.0827, 0.1140, 0.1924,   -inf],
        [0.0029, 0.0090, 0.0212, 0.0461, 0.0555, 0.0858, 0.7795]],
       grad_fn=<MaskedFillBackward0>)

In [21]:
# casual self attention
softmaxed_attention_weights = torch.softmax(masked_attention_weights, dim=1) #-inf alinca, 1'e dogru olasiligi artti
softmaxed_attention_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5316, 0.4684, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2718, 0.2414, 0.4867, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2262, 0.2372, 0.2706, 0.2660, 0.0000, 0.0000, 0.0000],
        [0.1783, 0.1764, 0.2449, 0.1912, 0.2092, 0.0000, 0.0000],
        [0.1720, 0.1582, 0.1663, 0.1600, 0.1651, 0.1785, 0.0000],
        [0.1193, 0.1200, 0.1215, 0.1246, 0.1257, 0.1296, 0.2593]],
       grad_fn=<SoftmaxBackward0>)

In [22]:
#drop out, bazi weightler cok buyur, buna reletive weightler cok kucuk kalir bu yuzden rastele bazi weightleri dropout ederiz

dropout_rate = 0.5
dropout = torch.nn.Dropout(dropout_rate)
dropout(softmaxed_attention_weights) 

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.9367, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5437, 0.4828, 0.9735, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4524, 0.0000, 0.5413, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3566, 0.3528, 0.4898, 0.0000, 0.4183, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.3326, 0.0000, 0.3301, 0.3570, 0.0000],
        [0.2386, 0.0000, 0.0000, 0.0000, 0.0000, 0.2592, 0.5187]],
       grad_fn=<MulBackward0>)

In [ ]:
#multihead, birden fazla baglamda inceleme. bu olmaz ise tek bi cumle icindeki baglama bakiyoruz.